# Health Retailers — Customer Segmentation (cahier §11.4)
## Complete, executable Data Preparation Lab

**Objective:** transform raw sales, transactions and customer files into a traceable, model-ready table describing
how each **identified customer** buys, then group customers into **homogeneous segments for marketing**.

**Analytical grain:** **one row = one identified customer** with at least one validated purchase.
The anonymous counter customer (most counter sales) is **excluded**: it is not a person and cannot be targeted.

**Data used:** the **real export only** by default (`INCLUDE_SYNTHETIC = False`): segments must describe real
behaviour. Setting it to `True` adds the generated months (tagged by date) for a technical test.

**Team division:** Person 1 = sales sources, cleaning, customer identification and joins. Person 2 = RFM features,
feature selection, transformation, PCA and clustering. Both validate the final dataset.

# 1 — Business objective

**Problem:** customer relations wants to know *who* the customers are and *how* they buy, to adapt offers: frequent
small baskets, occasional large orders, laboratories versus individuals, customers who stopped buying.

**Approach:** clustering (no target). The variables follow the **RFM** logic — **R**ecency, **F**requency,
**M**onetary — completed with tenure, product variety, discount and margin.

**Full workflow**
1. **Data Understanding:** profile each source.
2. **Data Cleaning:** encodings, Markdown pipes in names, decimal commas, dates, drafts, double counting, personal data.
3. **Data Transformation:** sale → transaction → company → customer file (join on **code**, never on name), aggregation
   to one row per customer.
4. **Feature Engineering:** RFM, tenure, active months, basket, variety, discount and margin rates, customer type.
5. **Feature Selection:** correlation-based removal of redundant variables.
6. **Dimensionality Reduction:** log transform, scaling, PCA.
7. **Optional clustering extension:** K-Means (elbow + silhouette), hierarchical clustering, segment profiles.
8. **Validation and reporting.**

## 0 — Setup and source files

| Source | Purpose | Key fields |
|---|---|---|
| `Documents.csv` / `documentItemm.csv` | Sales headers and lines | `Id`, `DocumentDate`, `DocumentType_Id`, `State`, `Transaction_Id` / `Quantity`, `PrixTotalHt` |
| `DocumentTypes.csv`, `RelatedDocuments.csv` | Sales vs returns, regrouped invoices | `Id` / `Document1_Id` |
| `Transactions.csv` | Which company each sale belongs to | `Id`, `Company_Id` |
| `CompaniesList.csv` | Company id ↔ code, customer or supplier | `Id`, `Code`, `CompanyType_Id`, `Name` |
| `Clients - …csv` | Customer type (individual / laboratory) | `Code`, `Nom` |
| `ProductStocks.csv`, `StockMovements.csv` | Product and cost of each line (margin) | `Id`, `Product_Id` / `DocumentId`, `Extra` |

In [1]:
# STEP 0: Import the libraries and define every input/output path in one place.
# Raw exports are read from the project folders and are never modified.
# All generated evidence (profiles, audits, matrices, figures) goes to outputs/customer_segmentation/.
# Change PROJECT if the health-retailers repository is stored elsewhere on your computer.

# Run once if needed:
# %pip install pandas numpy matplotlib scikit-learn

from pathlib import Path
import io, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# Locate the data branch checked out beside this branch, or set HEALTH_RETAILERS_DATA.
import os
_candidates = [Path(os.environ['HEALTH_RETAILERS_DATA']).expanduser()] if os.environ.get('HEALTH_RETAILERS_DATA') else []
for _base in (Path.cwd(), *Path.cwd().parents):
    _candidates.extend([_base / 'data', _base.parent / 'data'])
PROJECT = next((_p.resolve() for _p in _candidates if (_p / 'original').is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError('Check out the data branch beside ml-notebooks, or set HEALTH_RETAILERS_DATA to its root.')
RAW_DIR = PROJECT / 'original'                       # branch data-original: the 29 raw CSV files
SYNTH_DIR = PROJECT / 'generated'   # branch data-external: generated months
EXT_DIR = PROJECT / 'external'                        # weather and epidemics downloads
CLEAN_DIR = PROJECT / 'cleaned'                      # branch data-cleaned (product catalogue key)
OUTPUT_DIR = Path('outputs') / 'customer_segmentation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REAL_START = pd.Timestamp('2024-09-30')
REAL_END = pd.Timestamp('2025-04-30 23:59:59')       # last day of the real export
SYNTH_END = pd.Timestamp('2026-09-29 23:59:59')      # last day of the generated data
INCLUDE_SYNTHETIC = False                  # True = add the generated months (tagged, never mixed silently)

for folder in [RAW_DIR, SYNTH_DIR, EXT_DIR, CLEAN_DIR]:
    print(f'{str(folder):60s}', 'found' if folder.exists() else 'MISSING')

data\original found
data\generated found
data\external found
data\cleaned found


### Guided code explanation — Reading and conversion helpers

**Why this step exists:** the 29 raw files were not exported consistently. Some are UTF-8, others Windows-1252
or DOS code page 850; some use `;`, others `,`; amounts use a decimal comma; dates mix `MM/DD/YYYY` and
`DD/MM/YYYY`. One set of helpers guarantees that every file follows the same rules.

- `read_text()` tries UTF-8 first, then keeps the legacy code page that produces real French letters
  (`pièce` instead of `pi�ce`).
- `load_csv()` detects `;` or `,` from the header line and reads **every column as text** (`dtype='string'`),
  so identifiers and amounts are never converted by accident before we inspect them.
- `parse_amount()` turns `42,38` into `42.38`; unreadable values become `NaN` and are counted.
- `parse_bool()` maps `VRAI/FAUX`, `TRUE/FALSE`, `1/0` to one Boolean type.
- `parse_date()` uses an **explicit format** per column. We never let pandas guess between day-first and
  month-first, because `03/04/2025` is ambiguous.
- `profile()` returns the Data Understanding evidence: type, missing values, distinct values.

**What to tell your professor:** "Every conversion is explicit and audited: a value that cannot be parsed
becomes missing and is counted, it is never invented."

In [2]:
# Helpers shared by every step of the notebook.
# Every column is read as text first; conversions are explicit and audited later.

def read_text(path):
    raw = path.read_bytes()
    try:
        return raw.decode('utf-8-sig')
    except UnicodeDecodeError:
        # Legacy export: keep the code page that yields real French letters.
        candidates = [raw.decode(cp, errors='replace') for cp in ('cp1252', 'cp850')]
        score = lambda t: sum(t.count(c) for c in 'éèàçêù°') - 3 * sum(t.count(c) for c in 'Š‚ø')
        return max(candidates, key=score)

def load_csv(path):
    text = read_text(path)
    header = text.split('\n', 1)[0]
    sep = ';' if header.count(';') > header.count(',') else ','
    df = pd.read_csv(io.StringIO(text), sep=sep, dtype='string')
    df.columns = [str(c).strip() for c in df.columns]
    return df

def parse_amount(series):
    s = series.astype('string').str.strip().str.replace('\u00a0', '', regex=False).str.replace(' ', '', regex=False)
    return pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce').astype('float64')

def parse_bool(series):
    mapping = {'vrai': True, 'true': True, '1': True, 'faux': False, 'false': False, '0': False}
    return series.astype('string').str.strip().str.lower().map(mapping).astype('boolean')

def parse_date(series, fmt):
    return pd.to_datetime(series.astype('string').str.strip(), format=fmt, errors='coerce')

def profile(df):
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_count': df.isna().sum(),
        'missing_%': (100 * df.isna().mean()).round(2),
        'unique_values': df.nunique(dropna=True),
    })

def safe_left_merge(left, right, left_on, right_on, label):
    """Left join that refuses to multiply rows: the lookup key must be unique."""
    right = right.dropna(subset=[right_on])
    if right[right_on].duplicated().any():
        raise ValueError(f'{label}: lookup key {right_on} is not unique, aggregate it first')
    n = len(left)
    out = left.merge(right, how='left', left_on=left_on, right_on=right_on,
                     suffixes=('', '_' + label), validate='many_to_one')
    assert len(out) == n, f'{label} multiplied rows!'
    matched = out[right_on].notna().mean() * 100
    print(f'{label:28s} {n:>7,} rows kept | {matched:5.1f} % matched')
    return out

conversion_log = []
def convert(df, name, amounts=(), bools=(), dates=None):
    """Apply the conversions and record how many non-empty values could not be parsed."""
    df = df.copy()
    for col in amounts:
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_amount(df[col])
            conversion_log.append([name, col, 'amount', int(before - df[col].notna().sum())])
    for col in bools:
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_bool(df[col])
            conversion_log.append([name, col, 'boolean', int(before - df[col].notna().sum())])
    for col, fmt in (dates or {}).items():
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_date(df[col], fmt)
            conversion_log.append([name, col, f'date {fmt}', int(before - df[col].notna().sum())])
    return df

print('Helpers ready.')

Helpers ready.


## 1 — Data Understanding: profile every dataset separately

### Guided code explanation — Profiling

- `CompaniesList.csv` is Windows-1252 and its names are wrapped in Markdown pipes (`| PharmaNova France |`); one
  company (id 2, code `__part__`) has a lost name: it is the anonymous counter customer.
- `Clients.csv` writes the type inside the name: `1. Client particulier — Julien Moreau`.
- **Warning found during cleaning:** names were anonymised differently in each file (the same code has two different
  names), so the only reliable link is the **code**.
- E-mails and phone numbers are personal data: they are dropped right after profiling.

In [3]:
# STEP 1: Read each dataset independently.
CLIENTS = next(RAW_DIR.glob('Clients*.csv')).name
SOURCES = {'documents': 'Documents.csv', 'items': 'documentItemm.csv', 'related': 'RelatedDocuments.csv',
           'transactions': 'Transactions.csv', 'companies': 'CompaniesList.csv', 'clients': CLIENTS,
           'product_stocks': 'ProductStocks.csv', 'movements': 'StockMovements.csv'}
raw = {k: load_csv(RAW_DIR / f) for k, f in SOURCES.items()}
synth = {k: load_csv(SYNTH_DIR / SOURCES[k]) for k in ('documents', 'items', 'transactions', 'movements')} \
        if INCLUDE_SYNTHETIC else {}

overview = pd.DataFrame([[SOURCES[k], len(df), df.shape[1], int(df.duplicated().sum()),
                          int((df.isna().mean() == 1).sum())] for k, df in raw.items()],
                        columns=['file', 'rows', 'columns', 'exact_duplicates', 'empty_columns'])
display(overview)
overview.to_csv(OUTPUT_DIR / 'sources_overview.csv', index=False)
for key in ['companies', 'clients', 'transactions']:
    print(f'\n=== {SOURCES[key]} ===')
    display(raw[key].drop(columns=[c for c in raw[key] if c.lower() in ('email', 'e-mail', 'telephone')]).head(3))
    p = profile(raw[key]); display(p); p.to_csv(OUTPUT_DIR / f'profile_{key}.csv', index_label='column')

,file,rows,columns,exact_duplicates,empty_columns
0,Documents.csv,12542,35,0,2
1,documentItemm.csv,38705,34,0,10
2,RelatedDocuments.csv,864,4,0,0
3,Transactions.csv,12523,12,0,2
4,CompaniesList.csv,156,18,0,10
5,Clients - 24112025081756 (3).csv,93,11,0,1
6,ProductStocks.csv,6842,5,0,0
7,StockMovements.csv,49609,11,0,2



=== CompaniesList.csv ===


,Id,IsRemoved,CompanyType_Id,| Name |,IsGuest,Code,Id3,Id4,Id5,Id6,Id7,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,2,FAUX,1,|-------------------------------------------|,FAUX,__part__,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,4,FAUX,2,| PharmaNova France |,FAUX,FOU0001,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,5,FAUX,2,| HexaMed Distribution |,FAUX,FOU0002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


,dtype,missing_count,missing_%,unique_values
Id,string,0,0.0,156
IsRemoved,string,0,0.0,1
CompanyType_Id,string,0,0.0,2
| Name |,string,0,0.0,149
IsGuest,string,0,0.0,2
Code,string,0,0.0,156
Id3,string,156,100.0,0
Id4,string,156,100.0,0
Id5,string,156,100.0,0
Id6,string,156,100.0,0



=== Clients - 24112025081756 (3).csv ===


,Code,Nom,Adresse,Code postal,Ville,Pays,Site Web,Solde,Solde initial,Registre de commerce
0,CL0001,1. Client particulier — Julien Moreau,14 rue des Acacias,75001,Paris,France,<NA>,81850.1750,0,<NA>
1,CL0003,2. Client particulier — Claire Martin,22 avenue Victor Hugo,75002,Paris,France,<NA>,-1007.8790,0,<NA>
2,CL0004,3. Client particulier — Thomas Bernard,8 rue Lafayette,75003,Paris,France,<NA>,2908.3390,0,<NA>


,dtype,missing_count,missing_%,unique_values
Code,string,0,0.00,93
Nom,string,0,0.00,93
Adresse,string,0,0.00,91
Code postal,string,0,0.00,55
Ville,string,0,0.00,5
Pays,string,0,0.00,1
E-Mail,string,0,0.00,93
Site Web,string,92,98.92,1
Solde,string,0,0.00,64
Solde initial,string,0,0.00,1



=== Transactions.csv ===


,Id,IsPayed,PayedAmount,IsDestocked,IsRestocked,Company_Id,PaymentDocument_Id,RefundDocument_Id,DestockDocument_Id,RestockDocument_Id,BillingAddressId,ShippingAddressId
0,1,VRAI,"32,61",VRAI,FAUX,2,1,<NA>,1,<NA>,<NA>,<NA>
1,2,VRAI,"42,38",VRAI,FAUX,2,2,<NA>,2,<NA>,<NA>,<NA>
2,3,VRAI,"11,71",VRAI,FAUX,2,3,<NA>,3,<NA>,<NA>,<NA>


,dtype,missing_count,missing_%,unique_values
Id,string,0,0.00,12523
IsPayed,string,0,0.00,2
PayedAmount,string,0,0.00,7858
IsDestocked,string,0,0.00,2
IsRestocked,string,0,0.00,2
Company_Id,string,30,0.24,196
PaymentDocument_Id,string,1077,8.60,11446
RefundDocument_Id,string,12494,99.77,29
DestockDocument_Id,string,87,0.69,12436
RestockDocument_Id,string,12487,99.71,36


## 2 — Data Cleaning

### Guided code explanation — Cleaning customers and sales

- Company names: `str.strip(' |')` removes the pipes and padding; the header `| Name |` becomes `Name`.
- Personal data (`email`, `telephone`, `E-Mail`, address) is dropped: segmentation does not need it (GDPR minimisation).
- `Clients.Nom` is split with a regular expression into `customer_type` (`Client particulier`, `Laboratoire`) and the
  name.
- Sales lines follow the same rules as the other modules: customer flows only (sales `1, 5, 6`; returns `4, 7` in
  negative), drafts and invoices that regroup delivery notes excluded, comment lines quarantined.

In [4]:
# STEP 2: Clean customers, companies and sales lines.
companies = raw['companies'].copy()
companies.columns = [c.strip(' |') for c in companies.columns]
companies['Name'] = companies['Name'].str.strip(' |')
companies = convert(companies, 'companies', bools=['IsRemoved', 'IsGuest'])
companies = companies[['Id', 'Code', 'Name', 'CompanyType_Id', 'IsRemoved']]
print('Company 2 (anonymous counter customer):', companies.loc[companies.Id == '2', ['Code', 'Name']].values.tolist())

clients = raw['clients'][['Code', 'Nom']].copy()
parts = clients['Nom'].str.strip().str.extract(r'^\s*\d+\.\s*(?P<customer_type>[^—-]+?)\s*[—-]\s*(?P<name>.+)$')
clients = pd.concat([clients[['Code']], parts], axis=1)
clients['customer_type'] = clients['customer_type'].str.strip().map(
    {'Client particulier': 'INDIVIDUAL', 'Laboratoire': 'LABORATORY'}).fillna('UNKNOWN')
display(clients['customer_type'].value_counts().to_frame('customers in Clients.csv'))
assert clients['Code'].is_unique and companies['Id'].is_unique

def stack(key):
    return pd.concat([raw[key]] + ([synth[key]] if key in synth else []), ignore_index=True)
docs = convert(stack('documents'), 'documents', dates={'DocumentDate': '%m/%d/%Y %H:%M'})
items = convert(stack('items'), 'items', amounts=['Quantity', 'PrixHt', 'PrixTotalHt', 'SumDiscountTotal'])
transactions = stack('transactions')
moves = convert(stack('movements'), 'movements', amounts=['Quantity'])
assert docs['Id'].is_unique and items['Id'].is_unique and transactions['Id'].is_unique

SALE_TYPES, RETURN_TYPES = {'1', '5', '6'}, {'4', '7'}
grouped = set(raw['related']['Document1_Id']) & set(docs.loc[docs.DocumentType_Id == '1', 'Id'])
lines = items.merge(docs[['Id', 'DocumentDate', 'DocumentType_Id', 'State', 'Transaction_Id']], how='left',
                    left_on='Document_Id', right_on='Id', suffixes=('', '_doc'), validate='many_to_one')
lines['flag_comment_line'] = lines['ProductStock_Id'].isna() & lines['PrixTotalHt'].fillna(0).eq(0)
lines['flag_draft'] = lines['State'].ne('1')
lines['flag_grouped_invoice'] = lines['Document_Id'].isin(grouped)
customer_flow = lines['DocumentType_Id'].isin(SALE_TYPES | RETURN_TYPES)
display(lines.loc[customer_flow, ['flag_comment_line', 'flag_draft', 'flag_grouped_invoice']].sum().to_frame('lines'))
sales = lines[customer_flow & ~lines.flag_comment_line & ~lines.flag_draft & ~lines.flag_grouped_invoice].copy()
sign = np.where(sales['DocumentType_Id'].isin(RETURN_TYPES), -1, 1)
sales['qty'] = sign * sales['Quantity']
sales['revenue_ht'] = sign * sales['PrixTotalHt']
sales['discount_ht'] = sign * sales['SumDiscountTotal'].fillna(0)
sales['is_sale_document'] = sales['DocumentType_Id'].isin(SALE_TYPES)
print(f'Customer sales lines kept: {len(sales):,}')
display(pd.DataFrame(conversion_log, columns=['dataset', 'column', 'conversion', 'unparseable_non_null']))

Company 2 (anonymous counter customer): [['__part__', '-------------------------------------------']]


,customers in Clients.csv
customer_type,
LABORATORY,47
INDIVIDUAL,46


,lines
flag_comment_line,1024
flag_draft,927
flag_grouped_invoice,5638


Customer sales lines kept: 25,743


,dataset,column,conversion,unparseable_non_null
0,companies,IsRemoved,boolean,0
1,companies,IsGuest,boolean,0
2,documents,DocumentDate,date %m/%d/%Y %H:%M,0
3,items,Quantity,amount,0
4,items,PrixHt,amount,0
5,items,PrixTotalHt,amount,0
6,items,SumDiscountTotal,amount,0
7,movements,Quantity,amount,0


## 3 — Controlled Data Transformation and Joins

`documentItemm` → `Documents.Transaction_Id` → `Transactions.Company_Id` → `CompaniesList.Id` (code, customer or
supplier) → `Clients.Code` (type). Then the cost of each line (stock movement at validation) for the margin rate,
and finally **aggregation to one row per customer**.

### Guided code explanation — Joins and aggregation

- Each lookup is unique on its key (`safe_left_merge` checks it), so a sales line is never duplicated.
- Customers are companies with `CompanyType_Id = 1`, **except company 2** (anonymous counter customer).
- `customer_type` is `UNKNOWN` when the company code is absent from `Clients.csv`: it is **kept as unknown, never
  guessed** — this is the missing-value treatment of the categorical variable.
- `groupby('Company_Id').agg(...)` builds one row per customer: revenue, documents, quantities, lines, products,
  first and last purchase, discount, cost.

In [5]:
# STEP 3: Sale -> company -> customer, cost, aggregation per customer.
tx = transactions[['Id', 'Company_Id']].rename(columns={'Id': 'tx_id', 'Company_Id': 'company_id'})
sales = safe_left_merge(sales, tx, 'Transaction_Id', 'tx_id', 'sale -> company')
comp = companies.rename(columns={'Id': 'comp_id', 'Code': 'code', 'Name': 'company_name', 'CompanyType_Id': 'company_type'})
sales = safe_left_merge(sales, comp, 'company_id', 'comp_id', 'company -> code')
cli = clients.rename(columns={'Code': 'client_code', 'name': 'client_name'})
sales = safe_left_merge(sales, cli, 'code', 'client_code', 'code -> customer type')

ps = raw['product_stocks'][['Id', 'Product_Id']].rename(columns={'Id': 'stock_id', 'Product_Id': 'product_id'})
sales = safe_left_merge(sales, ps, 'ProductStock_Id', 'stock_id', 'stock line -> product')
val = moves[moves['Description'].str.startswith('Validation', na=False) & moves['DocumentId'].ne('0')].copy()
val['cump'] = pd.to_numeric(val['Extra'].map(
    lambda s: json.loads(s).get('cump') if isinstance(s, str) and s.startswith('{') else None), errors='coerce')
val['cost_key'] = val['DocumentId'] + '|' + val['ProductStock_Id']
cost = val.dropna(subset=['cump']).sort_values('Id').groupby('cost_key', as_index=False)['cump'].first()
sales['cost_key'] = sales['Document_Id'] + '|' + sales['ProductStock_Id']
sales = safe_left_merge(sales, cost, 'cost_key', 'cost_key', 'line -> unit cost')
sales['cost_ht'] = sales['qty'] * sales['cump']

walk_in = sales['company_id'].eq('2')
identified = sales[sales['company_type'].eq('1') & ~walk_in].copy()
print(f'Lines of the anonymous counter customer: {int(walk_in.sum()):,} '
      f'({100 * sales.loc[walk_in, "revenue_ht"].sum() / sales.revenue_ht.sum():.1f} % of revenue) — excluded')
print(f'Lines of identified customers: {len(identified):,}')

END = (SYNTH_END if INCLUDE_SYNTHETIC else REAL_END).normalize()
sale_docs = identified[identified.is_sale_document]
customers = identified.groupby('company_id').agg(
    code=('code', 'first'), name=('client_name', 'first'), company_name=('company_name', 'first'),
    customer_type=('customer_type', 'first'),
    revenue_ht=('revenue_ht', 'sum'), quantity=('qty', 'sum'), n_lines=('Id', 'count'),
    n_products=('product_id', 'nunique'), discount_ht=('discount_ht', 'sum'),
    cost_ht=('cost_ht', 'sum'), cost_known_ht=('revenue_ht', lambda r: r[identified.loc[r.index, 'cump'].notna()].sum()),
    first_purchase=('DocumentDate', 'min'), last_purchase=('DocumentDate', 'max'),
    active_months=('DocumentDate', lambda d: d.dt.to_period('M').nunique()))
customers['n_documents'] = sale_docs.groupby('company_id')['Document_Id'].nunique()
customers['customer_type'] = customers['customer_type'].fillna('UNKNOWN')
customers['name'] = customers['name'].fillna(customers['company_name'])
customers = customers[customers['n_documents'].fillna(0) > 0].reset_index()
print(f'Identified customers with at least one purchase: {len(customers)}')
display(customers['customer_type'].value_counts().to_frame('customers'))

sale -> company               25,743 rows kept | 100.0 % matched
company -> code               25,743 rows kept |  99.6 % matched
code -> customer type         25,743 rows kept |  34.8 % matched


stock line -> product         25,743 rows kept | 100.0 % matched


line -> unit cost             25,743 rows kept | 100.0 % matched
Lines of the anonymous counter customer: 16,617 (51.0 % of revenue) — excluded
Lines of identified customers: 9,020


Identified customers with at least one purchase: 60


,customers
customer_type,
INDIVIDUAL,28
LABORATORY,18
UNKNOWN,14


## 4 — Feature Engineering

| Axis | Feature | Meaning |
|---|---|---|
| Recency | `recency_days` | days since the last purchase (at the end of the period) |
| Frequency | `n_documents`, `purchases_per_month`, `active_months` | how often the customer buys |
| Monetary | `revenue_ht`, `avg_basket_ht` | how much, and per purchase |
| Tenure | `tenure_days` | days since the first purchase |
| Variety | `n_products`, `n_lines`, `quantity` | breadth of the basket |
| Commercial terms | `discount_rate_pct`, `margin_rate_pct` | discounts granted, margin earned |
| Type | `is_laboratory`, `type_unknown` | B2B vs individual, unknown kept explicit |

### Guided code explanation — Feature engineering

- `recency_days = END − last_purchase`: a large value = a customer who stopped buying.
- `purchases_per_month = n_documents / months between first purchase and END` (at least one month), so a recent
  customer is not penalised.
- `discount_rate_pct = discount / (revenue + discount)`: share of the gross price given away.
- `margin_rate_pct` is computed on the lines whose cost is known only, so a missing cost does not fake a 100 % margin.
- Customer type is encoded with **two indicators**: `is_laboratory` and `type_unknown`. Without the second one, an
  unknown customer would be silently treated as an individual.

In [6]:
# STEP 4: RFM and complementary features.
c = customers.copy()
c['recency_days'] = (END - c['last_purchase'].dt.normalize()).dt.days
c['tenure_days'] = (END - c['first_purchase'].dt.normalize()).dt.days
months_since_first = np.maximum(c['tenure_days'] / 30.44, 1)
c['purchases_per_month'] = c['n_documents'] / months_since_first
c['avg_basket_ht'] = c['revenue_ht'] / c['n_documents']
gross = c['revenue_ht'] + c['discount_ht']
c['discount_rate_pct'] = np.where(gross > 0, 100 * c['discount_ht'] / gross.where(gross > 0), 0)
c['margin_rate_pct'] = np.where(c['cost_known_ht'] > 0,
                                100 * (c['cost_known_ht'] - c['cost_ht']) / c['cost_known_ht'].where(c['cost_known_ht'] > 0), np.nan)
c['is_laboratory'] = c['customer_type'].eq('LABORATORY').astype('int8')
c['type_unknown'] = c['customer_type'].eq('UNKNOWN').astype('int8')
display(c[['code', 'name', 'customer_type', 'revenue_ht', 'n_documents', 'recency_days', 'tenure_days',
           'avg_basket_ht', 'n_products', 'discount_rate_pct', 'margin_rate_pct']].sort_values('revenue_ht', ascending=False).head(10).round(2))
display(c[['revenue_ht', 'n_documents', 'avg_basket_ht', 'recency_days', 'quantity']].describe().round(1))

,code,name,customer_type,revenue_ht,n_documents,recency_days,tenure_days,avg_basket_ht,n_products,discount_rate_pct,margin_rate_pct
51,CL0001,Julien Moreau,INDIVIDUAL,362407.65,852,0,212,425.36,1121,0.18,14.24
17,CL0006,Nicolas Petit,INDIVIDUAL,86025.11,1422,141,211,60.50,1301,18.17,18.44
7,CL0004,Thomas Bernard,INDIVIDUAL,8452.62,146,2,211,57.89,179,0.09,7.65
0,CL0003,Claire Martin,INDIVIDUAL,7721.23,50,13,211,154.42,198,6.15,10.61
33,CL0079,Laboratoire NovaMed Marseille,LABORATORY,3624.56,1,74,74,3624.56,81,6.76,11.47
19,CL0058,Laboratoire NovaBio Lyon,LABORATORY,2728.48,9,9,84,303.16,88,6.22,13.22
31,CL0073,Laboratoire NexaLab Marseille,LABORATORY,2560.55,4,11,96,640.14,34,2.12,33.26
47,CL0020,Chloé David,INDIVIDUAL,1786.34,12,5,196,148.86,30,19.38,12.68
49,CL0022,Emma Roux,INDIVIDUAL,1639.04,5,6,194,327.81,12,18.66,22.89
55,CL0027,Romain Bernard,INDIVIDUAL,1260.01,3,117,187,420.00,21,20.00,14.76


,revenue_ht,n_documents,avg_basket_ht,recency_days,quantity
count,60.0,60.0,60.0,60.0,60.0
mean,8226.1,43.9,234.9,83.1,359.8
std,47808.4,212.2,473.0,60.2,2242.4
min,33.4,1.0,31.2,0.0,1.0
25%,153.6,1.0,64.1,22.8,3.0
50%,272.1,2.0,127.7,78.0,5.5
75%,731.4,5.0,208.6,138.8,15.0
max,362407.6,1422.0,3624.6,181.0,17218.0


## 5 — Feature Selection

### Guided code explanation — Removing redundant variables

K-Means measures distances: two variables that say the same thing (for example `n_lines` and `quantity`) would count
that behaviour twice and pull the segments towards it.

- The correlation is computed on **log-transformed** amounts (Step 6 uses them), with Spearman (robust to outliers).
- Greedy rule: variables are examined in a **priority order** (RFM first); a variable is dropped if its |r| with an
  already kept variable exceeds **0.85**.
- `selection` records every decision and the variable it duplicates.

In [7]:
# STEP 5: Correlation-based selection, RFM variables first.
PRIORITY = ['recency_days', 'n_documents', 'revenue_ht', 'avg_basket_ht', 'tenure_days', 'purchases_per_month',
            'active_months', 'n_products', 'n_lines', 'quantity', 'discount_rate_pct', 'margin_rate_pct',
            'is_laboratory', 'type_unknown']
SKEWED = ['revenue_ht', 'avg_basket_ht', 'n_documents', 'n_products', 'n_lines', 'quantity', 'purchases_per_month']
view = c[PRIORITY].copy()
view[SKEWED] = np.log1p(view[SKEWED].clip(lower=0))
corr = view.corr(method='spearman')

kept, decisions = [], []
for var in PRIORITY:
    dup = next((k for k in kept if abs(corr.loc[var, k]) > 0.85), None)
    if dup is None and view[var].nunique() > 1:
        kept.append(var); decisions.append([var, True, ''])
    else:
        decisions.append([var, False, f'|r| = {abs(corr.loc[var, dup]):.2f} with {dup}' if dup else 'constant'])
selection = pd.DataFrame(decisions, columns=['feature', 'selected', 'reason_if_dropped'])
display(selection)
selection.to_csv(OUTPUT_DIR / 'feature_selection.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(corr)), corr.columns, rotation=90); ax.set_yticks(range(len(corr)), corr.columns)
fig.colorbar(im, ax=ax); ax.set_title('Customer features: Spearman correlation')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'feature_correlation.png', dpi=160); plt.show()

,feature,selected,reason_if_dropped
0,recency_days,True,
1,n_documents,True,
2,revenue_ht,True,
3,avg_basket_ht,True,
4,tenure_days,True,
5,purchases_per_month,True,
6,active_months,False,|r| = 0.93 with n_documents
7,n_products,True,
8,n_lines,False,|r| = 0.98 with n_products
9,quantity,False,|r| = 0.88 with n_products


C:\Users\firas\AppData\Local\Temp\ipykernel_42764\1746918939.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'feature_correlation.png', dpi=160); plt.show()


## 6 — Transformation, Scaling and PCA

1. **Missing values:** `margin_rate_pct` is missing when no line of the customer has a known cost → median.
2. **Log transform** `log(1 + x)` on skewed amounts and counts: a few large customers would otherwise dominate.
3. **StandardScaler:** every variable gets mean 0 and standard deviation 1, so days, euros and percentages weigh the
   same in the distance.
4. **PCA** keeping 90 % of the variance, plus a 2-component PCA to **draw** the customers on a plane.

Segmentation has no train/test split: all customers are described, and the transformations are fitted on all of them.

In [8]:
# STEP 6: Impute, log-transform, scale, PCA.
from sklearn.preprocessing import FunctionTransformer
log_cols = [v for v in kept if v in SKEWED]
plain_cols = [v for v in kept if v not in SKEWED]
preprocessor = ColumnTransformer([
    ('log', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('log1p', FunctionTransformer(lambda x: np.log1p(np.clip(x, 0, None)), feature_names_out='one-to-one')),
                      ('scale', StandardScaler())]), log_cols),
    ('plain', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), plain_cols),
], remainder='drop', sparse_threshold=0)
X = preprocessor.fit_transform(c)
names = preprocessor.get_feature_names_out()
assert np.isfinite(X).all()
pca = PCA(n_components=0.90, svd_solver='full')
Z = pca.fit_transform(X)
print('Matrix:', X.shape, '| PCA:', X.shape[1], '→', Z.shape[1],
      '| explained variance:', round(float(pca.explained_variance_ratio_.sum()), 3))
pca2 = PCA(n_components=2).fit(X)
Z2 = pca2.transform(X)
display(pd.DataFrame(pca2.components_.T, index=names, columns=['PC1', 'PC2']).round(2))
print('Variance of the 2-D view:', round(float(pca2.explained_variance_ratio_.sum()), 3))

Matrix: (60, 11) | PCA: 11 → 6 | explained variance: 0.911


,PC1,PC2
log__n_documents,0.46,-0.16
log__revenue_ht,0.45,0.01
log__avg_basket_ht,0.07,0.26
log__purchases_per_month,0.44,-0.07
log__n_products,0.45,0.01
plain__recency_days,-0.23,-0.40
plain__tenure_days,0.16,-0.53
plain__discount_rate_pct,-0.24,-0.14
plain__margin_rate_pct,-0.04,-0.05
plain__is_laboratory,-0.05,0.58


Variance of the 2-D view: 0.545


## 7 — Visual Evidence for the Presentation

In [9]:
# STEP 7: Effect of the log transform, PCA variance, customers on the PCA plane.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(c['revenue_ht'], bins=30); axes[0].set(title='Revenue per customer (raw)', xlabel='€ excl. VAT')
axes[1].hist(np.log1p(c['revenue_ht'].clip(lower=0)), bins=30); axes[1].set(title='After log(1 + x)', xlabel='log')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'log_transform.png', dpi=160); plt.show()

full = PCA().fit(X)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(1, X.shape[1] + 1), np.cumsum(full.explained_variance_ratio_), marker='o')
ax.axhline(0.9, linestyle='--', label='90 % threshold')
ax.set(xlabel='components', ylabel='cumulative explained variance', title='PCA variance'); ax.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'pca_variance.png', dpi=160); plt.show()

C:\Users\firas\AppData\Local\Temp\ipykernel_42764\1502472833.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'log_transform.png', dpi=160); plt.show()


C:\Users\firas\AppData\Local\Temp\ipykernel_42764\1502472833.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'pca_variance.png', dpi=160); plt.show()


## 8 — Final Validation and Deliverables

In [10]:
# STEP 8: Checks and exports.
assert c['company_id'].is_unique and not c['company_id'].eq('2').any()
assert len(X) == len(c)
c.to_csv(OUTPUT_DIR / 'customer_features.csv', index=False)
pd.DataFrame(X, columns=names).assign(company_id=c['company_id'].values).to_csv(OUTPUT_DIR / 'X_customers_scaled.csv', index=False)
pd.DataFrame(Z).assign(company_id=c['company_id'].values).to_csv(OUTPUT_DIR / 'X_customers_pca.csv', index=False)
summary = pd.DataFrame([
    ['identified_customers', len(c)], ['laboratories', int(c.is_laboratory.sum())],
    ['type_unknown', int(c.type_unknown.sum())],
    ['revenue_share_identified_%', round(100 * c.revenue_ht.sum() / sales.revenue_ht.sum(), 1)],
    ['features_selected', len(kept)], ['pca_components', Z.shape[1]],
], columns=['metric', 'value'], dtype=object)
display(summary)
summary.to_csv(OUTPUT_DIR / 'preparation_summary.csv', index=False)

,metric,value
0,identified_customers,60
1,laboratories,18
2,type_unknown,14
3,revenue_share_identified_%,48.3
4,features_selected,11
5,pca_components,6


# 9 — Optional extension: K-Means and hierarchical clustering

- **K-Means** for k = 2 … 8: the **elbow** (inertia) and the **silhouette** (how well each customer fits its own
  segment compared with the nearest other one; from −1 to 1, higher is better) guide the choice of k.
- **Hierarchical clustering (Ward)** with the same k as a second opinion; the **Adjusted Rand Index** (ARI) measures
  how much the two partitions agree (1 = identical, 0 = random).
- K-Means on the PCA components is compared the same way.

Segments are then **described with the original variables** (medians), so marketing can name them.

In [11]:
# STEP 9: Choose k, fit, compare, describe.
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

ks = range(2, 9)
scores = pd.DataFrame([[k, KMeans(k, n_init=20, random_state=42).fit(X).inertia_,
                        silhouette_score(X, KMeans(k, n_init=20, random_state=42).fit_predict(X))] for k in ks],
                      columns=['k', 'inertia', 'silhouette']).round(3)
display(scores)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(scores.k, scores.inertia, marker='o'); axes[0].set(title='Elbow (inertia)', xlabel='k')
axes[1].plot(scores.k, scores.silhouette, marker='o'); axes[1].set(title='Silhouette', xlabel='k')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'choose_k.png', dpi=160); plt.show()

best_k = int(scores.loc[scores[scores.k >= 3].silhouette.idxmax(), 'k'])    # at least 3 segments to be actionable
km = KMeans(best_k, n_init=50, random_state=42).fit(X)
c['segment'] = km.labels_
ward = AgglomerativeClustering(n_clusters=best_k, linkage='ward').fit_predict(X)
km_pca = KMeans(best_k, n_init=50, random_state=42).fit_predict(Z)
print(f'Chosen k = {best_k} | ARI K-Means vs Ward: {adjusted_rand_score(c.segment, ward):.2f} | '
      f'ARI K-Means full vs PCA: {adjusted_rand_score(c.segment, km_pca):.2f}')

profile_cols = ['revenue_ht', 'n_documents', 'avg_basket_ht', 'recency_days', 'tenure_days', 'n_products',
                'discount_rate_pct', 'margin_rate_pct']
seg = c.groupby('segment').agg(customers=('company_id', 'count'), revenue_share_pct=('revenue_ht', 'sum'),
                               laboratories=('is_laboratory', 'sum'), unknown_type=('type_unknown', 'sum'))
seg['revenue_share_pct'] = (100 * seg['revenue_share_pct'] / c['revenue_ht'].sum()).round(1)
seg = seg.join(c.groupby('segment')[profile_cols].median().round(1))
display(seg)
seg.to_csv(OUTPUT_DIR / 'segment_profiles.csv')
c[['company_id', 'code', 'name', 'customer_type', 'segment']].to_csv(OUTPUT_DIR / 'customer_segments.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 5))
for s, d in c.groupby('segment'):
    ax.scatter(Z2[d.index, 0], Z2[d.index, 1], s=40, alpha=0.8, label=f'segment {s} ({len(d)})')
ax.set(xlabel='PC1', ylabel='PC2', title='Customers on the first two PCA components'); ax.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'segments_pca.png', dpi=160); plt.show()

,k,inertia,silhouette
0,2,475.877,0.495
1,3,389.173,0.224
2,4,318.908,0.253
3,5,278.579,0.278
4,6,246.479,0.290
5,7,224.466,0.280
6,8,199.735,0.287


C:\Users\firas\AppData\Local\Temp\ipykernel_42764\4177532353.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'choose_k.png', dpi=160); plt.show()


Chosen k = 6 | ARI K-Means vs Ward: 0.87 | ARI K-Means full vs PCA: 0.84


,customers,revenue_share_pct,laboratories,unknown_type,revenue_ht,n_documents,avg_basket_ht,recency_days,tenure_days,n_products,discount_rate_pct,margin_rate_pct
segment,,,,,,,,,,,,
0,4,94.1,0,0,47238.9,499.0,107.5,7.5,211.0,659.5,3.2,12.4
1,5,0.1,0,2,134.6,1.0,134.6,170.0,170.0,1.0,16.7,33.3
2,17,2.4,0,0,723.2,5.0,121.5,68.0,194.0,8.0,19.4,15.9
3,16,0.7,15,0,231.9,2.0,72.9,63.0,108.5,3.0,20.0,15.7
4,12,0.5,0,12,156.4,1.0,156.4,132.0,132.0,2.5,20.0,13.3
5,6,2.1,3,0,1635.6,1.0,545.9,17.0,55.0,20.5,13.4,12.8


C:\Users\firas\AppData\Local\Temp\ipykernel_42764\4177532353.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'segments_pca.png', dpi=160); plt.show()


### Reading the segments

Name each segment from its medians, for example: *high-value regulars* (many documents, low recency),
*occasional large orders* (few documents, large basket), *lapsed customers* (high recency), *small individuals*.
A silhouette below about 0.25 means the segments overlap: present them as tendencies, not as hard groups.
The ARI tells whether another method (Ward) or the PCA representation would give the same grouping; a high ARI makes
the segmentation more credible.

**Watch the type indicators.** After scaling, the binary variables `is_laboratory` and `type_unknown` create large
distances, so some segments may simply be "all laboratories" or "all unknown type". If that happens, a cleaner
alternative is to cluster on **behaviour only** (remove the two indicators from `kept`) and then cross the segments
with the customer type in a table.

## 10 — What to say during the presentation

**Person 1:** "We linked every validated sale to its company through the transaction, identified customers by their
code because names were anonymised differently in each file, excluded the anonymous counter customer and computed the
cost of each line to obtain a margin."

**Person 2:** "We built RFM variables plus tenure, variety, discount and margin, removed redundant variables with a
correlation rule, log-transformed skewed amounts, standardised everything and used a PCA to visualise and test the
segments. K-Means was chosen with the elbow and the silhouette, and checked against a hierarchical clustering."

### Limitations to disclose
- A few dozen identified customers over 7 months: segments are indicative.
- Customer type is unknown for part of the customers (kept as `UNKNOWN`).
- Recency depends on the end date of the period analysed.
- Most counter sales are anonymous and therefore outside this segmentation.